# w9_swin.ipynb — 滑窗新鲜锚场 swin{W}step{S}loop{L}i2ce

User design: 锚目录作为环序列,窗口在其上滑动充当"过去的锚包",但**永不
缓存** — 每步 L 个微通道各自新鲜编码下一段 W 个游戏,与永远在场的当前
batch 锚拼成本通道的 CE 分区,逐通道立即 backward(窗口激活随手释放 =
计算量换显存);指针每步前进 S*W。bkq/bkb 的尸检证明缓存快学生的行必然
场不连贯;bce 证明全新鲜小场存活但要付覆盖税 — 本设计把覆盖率变成
(bs+L*W)/N 的连续旋钮,且锚(质心正样本)保留、noname 梯度边全开。

SEMANTICS (user final): step{S} = 每微通道滑动的游戏数(丢 S 个旧的、引
S 个新的);通道 l 的窗口起点 = p + l*S,相邻通道重叠 W-S;步末 p += L*S。
原始设想 step=1(逐游戏滑动,绕环 2 圈 = 3260 次反传/步)被用户否决为
太贵;终案 **swin168step84loop2i2ce @4096** — 半窗滑动链,每环位恰被
2 通道覆盖,全环 9.6 步一扫,每步覆盖 (192+252)/1613 ≈ 27.5%,预估
~25-30G(全价 45G)。Refs: i2ce@4096 (.750) / i2q2ce@4096 (.716) /
i2esce@4096 (tag .741) / i2ce@2048。ZS-only, rvsel。AUTO-STOPS。

**COVERAGE SCAN (user 2026-07-19)**: swin84s42L2 (19.7%/step) 与
swin336s168L2 (43.2%/step) 入列 — 与已出分的 swin168 (27.5%, non .691
tag .732/.733) 和全价 i2ce@4096 (100%, .706/.696) 四点画出
coverage→(non, tag) 曲线,钉死饱和膝盖的位置。半窗滑动链语义不变
(S=W/2, L=2, 每环位 2 通道/圈)。


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # fixed-split campaign dir

# (arm, total anchor cap)
JOBS = [("wcle_swin168step84loop2i2ce_icetf", 4096),   # 27.5%/step (done)
        ("wcle_swin84step42loop2i2ce_icetf", 4096),    # 19.7%/step scan-low
        ("wcle_swin336step168loop2i2ce_icetf", 4096)]  # 43.2%/step scan-high
EPOCHS = 2000
os.makedirs(OUT_DIR, exist_ok=True)
print("towers:", [f"w9_{a}_g{c}" for a, c in JOBS], f"@ {EPOCHS}ep")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run the pending towers (round-robin over GPUs; several fit per 80G card).
# ZS-only done marker = ep{EPOCHS} npz.
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
todo = []
for arm, cap in JOBS:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} done"); continue
    todo.append((arm, cap, nm))
print(f"{len(todo)} tower(s) to run")

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, arm, cap, nm):
    if not J.try_claim(cdir, nm):
        # corpse window: a pod that died <120s ago still looks alive.
        # Wait out DEAD_SEC once and retry before giving up (fast relaunch
        # otherwise skips everything and auto-stops -- looks like a crash).
        print(f"[claim] {nm} fresh/held -- waiting 130s for the corpse "
              "window, then retrying once", flush=True)
        time.sleep(130)
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
           "--epochs", str(EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
           "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
           "--topup-seeds", str(J.TOPUP_SEEDS),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm}", flush=True)
    t0 = time.time()
    with open(logd / f"{arm}_g{cap}.log", "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
    if p.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

ths = [threading.Thread(target=run_one, args=(gpus[i % len(gpus)], arm, cap, nm))
       for i, (arm, cap, nm) in enumerate(todo)]
for i, t in enumerate(ths):
    if i:
        time.sleep(120)   # stagger starts: each worker's data-load phase has
        # an ~8.5G host-RAM transient (POOL np.load) + array loads; three
        # simultaneous startups stack the peak -- suspect in the container-OOM
        # pod deaths (attempt1 died at ~ep50, right after triple load).
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: swin vs the 4096 anchor-supply pantheon (ZSbest-primary).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_swin84step42loop2i2ce_icetf_g4096", "swin84s42L2 @4096 (19.7%)"),
        ("wcle_swin168step84loop2i2ce_icetf_g4096", "swin168s84L2 @4096 (27.5%)"),
        ("wcle_swin336step168loop2i2ce_icetf_g4096", "swin336s168L2 @4096 (43.2%)"),
        ("wcle_i2ce_icetf_g4096", "i2ce@4096 (full price)"),
        ("wcle_i2q2ce_icetf_g4096", "i2q2ce@4096 (soft-band I)"),
        ("wcle_i2esce_icetf_g4096", "i2esce@4096 (EMA gallery)"),
        ("wcle_i2ce_icetf_g2048", "i2ce@2048 (joint ref)"),
        ("wcle_i2ce_icetf", "i2ce@512 (healthy ref)")]
def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ROWS:
    print(_row(lab, f"w9_{arm}"))


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)